# Module 3: Finding Failure Modes

> Self-directed module, ~30 min. LangSmith UI focused — no code cells in this notebook.

You have traces from Module 2's warm-up runs. These tools work on any trace corpus — return here after completing Modules 04–06 to layer in evaluation scores and annotation data.

This module covers how to turn raw trace data into signal that drives agent improvements — without manually reading every trace.

## The shift in scale

Manually reviewing traces was a viable workflow when agents were short and few. Three trends have pushed past that point:

- **Agents are longer-running.** A single trace can span dozens of tool calls and multiple subagent invocations. Even a careful reviewer needs minutes per trace; scanning hundreds is impractical.
- **Multimodal inputs and outputs are becoming common.** Traces include images, audio, attachments, and structured documents alongside text. Skimming for a pattern means inspecting more than text spans.
- **Agents are proliferating.** A typical enterprise deployment runs many agents, each producing its own trace stream. Even with focused attention, surface area outpaces a single reviewer.

LangSmith ships three features built around this problem. Each turns a different slice of trace data into action:

- **Chat** — a workspace AI assistant. Ask natural-language questions, get answers grounded in your project's traces, threads, prompts, and experiments.
- **Insights Agent** — an automated job that examines a trace corpus and produces hierarchical category reports. Surfaces patterns nobody knew to look for.
- **Engine** — a closed-loop system that detects recurring failures, diagnoses root causes, and proposes fixes as pull requests. Currently available on cloud; self-hosted support is on the roadmap.


---

## Engine

> Engine needs to be enabled in your LangSmith org. If it's not visible in your project, it needs to be enabled at the organization level, which can be done by your LangSmith administrator. Engine is currently available on cloud; self-hosted LangSmith support is coming.

### What Engine does

Engine plugs into the full agent engineering lifecycle: trace → recurring failure detected → root cause diagnosed → fix proposed → evaluator deployed → dataset examples generated. It scans your project's traces every 6 hours by default and surfaces issues without requiring manual review.

For each detected issue, Engine surfaces the relevant traces, proposes a fix, generates a custom evaluator to prevent regressions, and creates ground truth dataset examples from the production trace inputs for offline evaluation.

Where Insights gives you a categorized view of what's happening, Engine produces specific, actionable issues with proposed fixes — code or prompt modifications you can apply via a pull request, plus suggested evaluators to catch regressions on the same failure mode in future iterations. Insights surfaces *what's happening*; Engine surfaces *what needs to change*.

Engine uses LangChain-managed inference exclusively — Bring Your Own Key (BYOK) is not supported.

### Enabling Engine

Two steps — org-level then per-project:

1. **Organization Admin** enables Engine workspace-wide in **Settings → Engine → Engine enablement**. Enabling requires acknowledging that trace data will be processed using LangChain-managed inference.

2. **Per project**, navigate to the **Engine tab** of your tracing project and configure:
   - Optionally connect a **GitHub repository** — recommended. Engine uses your source code to diagnose problems more accurately and to open pull requests directly from issues. Only repositories the LangSmith GitHub app can access are shown.
   - Select **priority categories** — which failure types to focus on (e.g., Tool Call Failures, Latency). Add custom concerns with **+ Add something specific**.
   - Click **Start Analyzing**. The first run can take up to 20 minutes.

3. **Review the agent overview document.** Before surfacing issues, Engine generates a document describing your project's purpose, architecture, and key metrics based on your traces. Review and edit it before continuing — Engine uses it as context for all analysis, so accuracy here directly affects issue quality. Update it at any time from Engine settings as your application evolves.

Once setup completes, Engine populates an **Issues** list in the project. Use the filter and sort controls at the top of the list to focus by priority, status, or severity.

<img src="../images/engine_config.png" alt="Engine configuration dialog with GitHub connection and priority categories" style="width: auto; max-height: 380px; border-radius: 8px;">

### Reviewing an issue

Click any issue in the list to open its detail panel. At the top, a **diagnosis** describes the problem and its impact, backed by the traces Engine linked to it.

The detail panel has four sections:

- **Linked traces** — the specific traces that support the diagnosis. Click any trace to inspect it directly.
- **Proposed Fix** — suggested code or prompt changes targeting the root cause. Quality is higher when a GitHub repository is connected, since Engine can reference your actual source.
- **Suggested Evaluator** — a ready-to-use evaluator you can deploy as a run rule to catch this failure mode in future traces. If the evaluator fires after you close the issue, Engine automatically reopens it.
- **Offline Examples** — dataset examples generated from the production trace inputs that triggered the issue. Each example shows the input, the wrong output the agent produced, and a proposed expected output structured as named **assertions** — short claims about what a correct response should or shouldn't include.

<img src="../images/engine_issue.png" alt="Engine issue detail with diagnosis, proposed fix, and suggested evaluator" style="display: block; width: 67%; border-radius: 8px; margin-bottom: 1rem;">

### Taking action

From the issue detail panel:

- **Open PR** — opens a GitHub pull request in your connected repository with the proposed fix already applied. The button becomes **View PR** once the PR is open. Supported for repos built with Deep Agents, LangChain, and LangGraph.
- **Create Evaluator** — deploys the suggested evaluator as an online evaluator. Configure the name, run filters, and sampling rate; toggle **Apply to past runs** to see how many historical traces it would have flagged before committing. See Module 5 for the evaluator model.
- **Add offline examples** — adds the proposed examples to a dataset. Optionally route through an annotation queue to review and edit the assertions before committing. See Module 4 for the dataset and experiment loop.
- **Copy Fix Context** — copies a prompt with the full issue details to your clipboard. Use it directly with an LLM or coding assistant to work through the fix outside of LangSmith.
- **Resolve / Ignore** — mark as fixed or dismiss. Providing a reason feeds back into Engine's analysis to improve future scans. If a resolved issue resurfaces in new traces, Engine automatically reopens it.

<img src="../images/engine_triage.png" alt="Engine main triage issue view" style="display: block; width: auto; max-height: 420px; border-radius: 8px; margin-bottom: 1rem;">

Priority adjustments and feedback (accept / reject / refine) train Engine's understanding of which issues matter for this specific project — relevance improves over iterations.

---

## Chat

> Chat needs to be enabled in your LangSmith org. If it's not visible in your instance, it needs to be enabled at the organization level, which can be done by your LangSmith administrator.

Chat is an AI assistant embedded directly in the LangSmith workspace. It lives in the bottom-right corner of every observability, prompt engineering, and evaluation page and answers natural-language questions about whatever you're currently looking at — traces, threads, prompts, experiments, datasets, queues.

Chat accesses the same data the UI does. The value isn't new visibility — it's skipping the steps of writing a filter expression, scrolling a long list, or opening several traces to compare them. For ad-hoc questions during development or review, it's the fastest path.

<img src="../images/chat.png" alt="Chat assistant panel open in the LangSmith UI" style="width: auto; max-height: 380px; border-radius: 8px;">

### Accessing Chat

- **Keyboard shortcut:** `Cmd+I` (Mac) or `Ctrl+I` (Linux / Windows) — toggles the panel from any page.
- **Clear thread:** `Cmd+Shift+O` / `Ctrl+Shift+O` — starts a fresh conversation.
- **Click** the Chat icon in the bottom-right corner.

### Questions worth asking against the retail assistant project

After the Studio testing in Module 7, try these in Chat. Each one assumes you're on a specific LangSmith page — open that page first, then ask Chat the question.

- **On the tracing project page** (the one with your warm-up + Studio traces — open any individual trace first): *"Summarize the tool calls that the agent made in this trace."* — fast way to read a multi-step trace without expanding every span.
- **On the tracing project page, after Module 5 has set up the correctness evaluator:** *"Which runs failed the correctness eval? Summarize why."* — Chat reads the judge's comment field and clusters the failures. Skip this one if you haven't run Module 5.
- **On the tracing project page:** *"Did any customers get an unknown-order or unknown-account response?"* — surfaces the `ORD-99999` / `CUST-555` queries and any similar lookups.
- **On the tracing project page:** *"What's the latency distribution for runs that delegated to the product-recommendation subagent?"* — answers questions that would otherwise require export + analysis.
- **On the dataset's Experiments tab (Module 4):** *"Compare this experiment to the previous one. What changed?"* — the page context scopes Chat to the experiments you're looking at.

Chat can also edit prompts conversationally (e.g., *"make the judge prompt stricter on grounding in tool results"*) — useful in the Prompt Hub. The conversational surface stays consistent; what changes is the page you're on, which scopes what Chat sees.

---

## Insights Agent

> Insights needs to be enabled in your LangSmith org. If Insights reports aren't visible in your project, it needs to be enabled at the organization level, which can be done by your LangSmith administrator.

Insights is an automated analysis job that runs against a project's trace corpus and groups the runs into a hierarchy of categories — top-level themes (e.g., "order status requests", "product recommendations", "unknown identifiers") with nested subcategories. The output is an executive summary, distribution bars showing how many runs fell into each category, and per-category aggregates (latency, error rate, cost, evaluator feedback).

This is the right tool for the question "what is actually happening in production?" Patterns nobody thought to look for surface as their own categories. Use Insights to find what's worth investigating; use Chat to dig into specific traces once you know where to look.

### Prerequisites

**A healthy trace corpus.** By this point in the modules, the warm-up in Module 2, the experiments in Module 4, the trigger prompts in Modules 5 and 6, and (optionally) the Studio testing in Module 7 will have collectively produced enough trace volume for Insights to find patterns. Production deployments accrue traces continuously and don't need a manual seed.

### Configuring a report

1. Open the **Tracing Projects** page and select the project (`langsmith-poc-retail-assistant` if you kept the default).
2. Click **+New → New Insights Report**.
3. Use the defaults unless you have a specific configuration you want to test — the report will auto-generate a category structure from your trace data.
4. Launch. The job runs in the background and typically completes within 30 minutes.

<img src="../images/insights_config.png" alt="Insights Report configuration dialog" style="width: auto; max-height: 380px; border-radius: 8px;">


### Reading the report

The report opens to an executive summary — the largest patterns by run share, with one-sentence descriptions. From there:

- **Click a category** to drill into its subcategories and individual sample traces.
- **Distribution bars** show how many runs fell into each category, sorted by frequency.
- **Per-category aggregates** show error rate, latency percentiles, cost, and any attached evaluator feedback. A category with high frequency *and* high error rate is the first place to look for a regression.
- **Sample traces** are linked directly — clicking opens the trace in the standard trace tree view.

<img src="../images/insights_report.png" alt="Insights report with hierarchical categories and distribution bars" style="width: auto; max-height: 420px; border-radius: 8px;">


### Scheduling

Reports can be set to re-run on a schedule — daily, weekly, or a custom cron expression. Use the defaults unless you have a specific cadence in mind.


### Acting on Insights findings

Insights surfaces patterns; the loop closes via the earlier modules. When a category looks actionable:

- **High-frequency failure category** → promote sample traces to an annotation queue (Module 6) for human labeling. The labeled examples become eval data for the next iteration.
- **Latency outlier category** → open a sample trace and use **Chat** to summarize the trace tree, then tune the offending tool or prompt.
- **Ungrounded-answer pattern** → tighten the judge prompt (Module 5) to score grounding in tool results more strictly.

---

## Putting it together

A practical workflow for an enterprise POC:

1. **Generate trace volume.** Trace volume accrues across the modules as you run the warm-ups, experiments, and trigger prompts. Scripted batch runs or real production traffic work too. All three tools need enough traces to find patterns.
2. **Engine short-circuits the discovery cycle** for recurring failures: it surfaces each issue with a proposed fix, a suggested evaluator, and dataset examples ready for experiments. If Engine is enabled, check the Issues list before doing manual discovery with Chat or Insights.
3. **Use Chat during development and review** for ad-hoc questions — slowest runs this morning, what failed correctness, what changed between experiments. Low setup, immediate.
4. **Configure an Insights report** to surface patterns across the trace corpus. Review the executive summary to catch behavioral drift.
5. **When an Insights category looks actionable** — high-frequency failures, latency outliers, missed citations — promote affected traces to an annotation queue (Module 6) or a dataset (Module 4). The labeled / curated examples drive the next iteration of the judge, the agent, or both.

## Recap

| Tool | When to use | Self-hosted |
|---|---|---|
| **Engine** | Closed-loop detection, fix proposal, and dataset generation. | Coming soon. |
| **Chat** | Ad-hoc questions during development or review. Lowest setup. | Confirm with workspace admin. |
| **Insights Agent** | Automated periodic discovery of patterns across a trace corpus. | Yes. |

All three convert raw trace data into signal worth acting on. Return here after completing Modules 04–06 to use Chat and Insights against a richer trace corpus that includes evaluation scores and annotation queue feedback.

Next: `04_datasets_and_experiments.ipynb` — define a fixed dataset and run offline experiments against it.